In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

def generate_data(n_samples, n_features, random_state=None):
    X, y = make_classification(n_samples, n_features, n_classes=2, random_state=random_state)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)
    train_loader = get_dataloader(X_train, y_train)
    test_loader = get_dataloader(X_test, y_test, shuffle=False)
    return train_loader, test_loader


def get_dataloader(X, y, batch_size=32, shuffle=True):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)
    # 将多个tensor组合成一个整体，形成一个可以迭代的数据集。
    dataset = TensorDataset(X_tensor, y_tensor)
    # 将Dataset放到Dataloader当中，这样就可以分批次获取数据。
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return dataloader


class BinaryModel(nn.Module):
    """用于二分类的神经网络类。
    """
    def __init__(self, in_features, out_features=1):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        x = self.linear(x)
        # x = F.sigmoid(x)
        return x

n_samples = 1000
n_features = 5
learning_rate = 0.005
epochs = 30
train_loader, test_loader = generate_data(n_samples, n_features)
model = BinaryModel(n_features)
# 定义损失函数，二分类使用二元交叉熵损失函数。
# BCELoss与BCEWithLogitsLoss的区别：
# 前者的输入为概率，后者的输入为logits。后者相当于是Sigmoid + BCELoss。
# 官方推荐使用后者。
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# 训练与评估模型。
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    for X_batch, y_batch in train_loader:
        output = model(X_batch).flatten()
        loss = criterion(output, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        # 计算正确率。
        y_pred = (output > 0).float()
        train_correct += (y_pred == y_batch).sum().item()
    train_loss /= len(train_loader)
    train_accuracy = train_correct / len(train_loader.dataset)

    model.eval()
    test_loss = 0.0
    test_correct = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            output = model(X_batch).flatten()
            loss = criterion(output, y_batch)
            test_loss += loss.item()
            y_pred = (output > 0).float()
            test_correct += (y_pred == y_batch).sum().item()
    
    test_loss /= len(test_loader)
    test_accuracy = test_correct / len(test_loader.dataset)

    print(f"Epoch {epoch + 1} / {epochs} "
        f"Train Loss:{train_loss:.4f} Train Accuracy: {train_accuracy:.4f} "
        f"Test Loss:{test_loss:.4f} Test Accuracy: {test_accuracy:.4f}")

Epoch 010, Loss: 0.5553, Accuracy: 0.7929
Epoch 020, Loss: 0.4840, Accuracy: 0.8643
Epoch 030, Loss: 0.4271, Accuracy: 0.8857
Epoch 040, Loss: 0.3822, Accuracy: 0.9000
Epoch 050, Loss: 0.3475, Accuracy: 0.9143
Epoch 060, Loss: 0.3213, Accuracy: 0.9143
Epoch 070, Loss: 0.3020, Accuracy: 0.9143
Epoch 080, Loss: 0.2875, Accuracy: 0.9214
Epoch 090, Loss: 0.2766, Accuracy: 0.9214
Epoch 100, Loss: 0.2681, Accuracy: 0.9214
Test Accuracy: 0.9166666865348816


In [ ]:
class MultiModel(nn.Module):
    """用于多分类的神经网络类。
    """
    def __init__(self, in_features, out_features=1):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        x = self.linear(x)
        # x = F.log_softmax(x)
        return x


n_samples = 1000
n_features = 5
learning_rate = 0.01
epochs = 30
train_loader, test_loader = generate_data(n_samples, n_features)
model = MultiModel(n_features, 3)
# 定义损失函数，多分类使用交叉熵损失函数。
# NLLLoss与CrossEntropyLoss的区别：
# 前者的输入为对数概率（经过LogSoftmax处理后的结果），后者的输入为logits。后者相当于是LogSoftmax + NLLLoss。
# 官方推荐使用后者。
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# 训练与评估模型。
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    for X_batch, y_batch in train_loader:
        output = model(X_batch)
        loss = criterion(output, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        # 计算正确率。
        y_pred = output.argmax(dim=1)
        train_correct += (y_pred == y_batch).sum().item()
    train_loss /= len(train_loader)
    train_accuracy = train_correct / len(train_loader.dataset)

    model.eval()
    test_loss = 0.0
    test_correct = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            output = model(X_batch)
            loss = criterion(output, y_batch)
            test_loss += loss.item()
            y_pred = output.argmax(dim=1)
            test_correct += (y_pred == y_batch).sum().item()
    
    test_loss /= len(test_loader)
    test_accuracy = test_correct / len(test_loader.dataset)

    print(f"Epoch {epoch + 1} / {epochs} "
        f"Train Loss:{train_loss:.4f} Train Accuracy: {train_accuracy:.4f} "
        f"Test Loss:{test_loss:.4f} Test Accuracy: {test_accuracy:.4f}")

class MultiModel(nn.Module):
    """用于多分类的神经网络类。

    使用多层网络结构（增加隐藏层）。
    """
    def __init__(self, in_features, out_features=1):
        super().__init__()
        n_hidden = 64
        self.hidden_layer = nn.Linear(in_features, n_hidden)
        self.output_layer = nn.Linear(n_hidden, out_features)


    def forward(self, x):
        x = self.hidden_layer(x)
        x = F.relu(x)
        x = self.output_layer(x)
        return x